# Limpeza e Tratamento Inicial
### O mercado de shows em São Paulo (2021–2025)

##### O projeto pretende entender o comportamento e fluxo de público em shows médio e grande em São Paulo, considerando a popularidade do nome no spotify, o genero qdo nome, o local onde ocorrem os shows, valores de ingresso e data

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import random
import math
import re

from ydata_profiling import ProfileReport
import chardet
import unicodedata
import time

from spotipy import Spotify
from spotipy.oauth2 import SpotifyClientCredentials

import json
import base64
from dotenv import load_dotenv

import matplotlib.pyplot as plt
import seaborn as sns
from phik.report import plot_correlation_matrix

# Código para padronizar nomenclaturas

In [ ]:
def normalize(nomes):
    if pd.isna(nomes):
        return None
    nomes = str(nomes)
    
    nomes = nomes.replace("’", "'")
    nomes = nomes.replace("‘", "'") 
    nomes = nomes.replace("“", '"') 
    nomes = nomes.replace("”", '"')  
    nomes = nomes.replace("–", "-")  
    nomes = nomes.replace("—", "-") 
    nomes = nomes.replace("‐", "-") 
    
    
    nomes = unicodedata.normalize('NFKD', nomes)
    nomes = ''.join(c for c in nomes if not unicodedata.combining(c))

    nomes = re.sub(r'\s+', ' ', nomes).strip().upper()
    
    return nomes

# Análise inicial das bases de dados criadas

## Base de dados de locais

In [ ]:
with open('../data/external/locais.csv', 'rb') as f:
    enc = chardet.detect(f.read())
    print(enc)

locais = pd.read_csv('../data/external/locais.csv', sep= ';', encoding= 'iso8859')

print(locais.info())
locais.head()

In [ ]:
# Como a criação dessa base foi manual no excel, para os valores de latitude e Longitude materem o formato útil inclue o ', que será retirado aqui
locais['Latitude'] = locais['Latitude'].str.lstrip("'")
locais['Longitude'] = locais['Longitude'].str.lstrip("'")

locais['nome_local'] = locais['Nome_Local'].apply(normalize)
locais['Categoria_Local'] = locais['Categoria_Local'].apply(normalize)
locais['Tipo_Espaco'] = locais['Tipo_Espaco'].apply(normalize)
locais['Bairro'] = locais['Bairro'].apply(normalize)
locais['Capacidade'] = locais['Capacidade']*1000

locais = locais.drop(['Nome_Local'], axis = 1)

pr_locais = ProfileReport(locais, title='Profiling Report of Locais Data Base')
pr_locais

# Como essa base de dados foi construída manualmente, foram tomados todos os cuidados para não haver dados em branco nela, tentando mantê-la de forma mais limpa e com os dados o mais claros e simplificados possível. Assim, apenas criarei a matriz de correlação para decisão de redução e  limpeza da base.

In [ ]:
# Como são muitas variáveis categóricas utilizarei o método de análise phik

phik_corr = locais.phik_matrix(interval_cols=[])

plt.figure(figsize=(14, 12))
sns.heatmap(
    phik_corr,
    annot=True,
    fmt='.2f',
    cmap='crest',
    center=0,
    linewidths=2,
    linecolor='#FFFFFF'
)
plt.title('Matriz de Correlação (Phik)', fontsize=14)
plt.show()

# Para manter a base mais 'clean' considerando a matriz anterior, excluirei bairro e local_id, considerando que são variáveis duplicadas. Posteriormente excluirei também a Capacidade, após fazer cálculos para construção da base final.

locais = locais.drop(['Local_ID', 'Bairro'], axis = 1)

locais.to_csv('../data/raw/locais_limpo.csv', index=False)

## Base de dados inicial de shows

In [ ]:
with open('../data/external/shows.csv', 'rb') as f:
    enc = chardet.detect(f.read())
    print(enc)

shows = pd.read_csv('../data/external/shows.csv', encoding='utf-8')
print(shows.info())
shows.head()

In [ ]:
# Transformar a data para o formato correto
shows['data'] = pd.to_datetime(shows['data'], errors='coerce')

shows['nome_local'] = shows['local'].apply(normalize)
shows['nome'] = shows['artista'].apply(normalize)

# Existem dados em branco apenas na coluna de local em 15 linhas
shows[shows['local'].isnull()]

# Substituição dos valores nulos pela moda em relação acoluna artista
fill_mode = lambda x: x.fillna(x.mode()[0])
shows['local'] = shows.groupby('artista')['local'].transform(
    lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else 'Desconhecido')
)
shows.info()

In [ ]:
pr_shows = ProfileReport(shows, title='Profiling Report of Shows Data Base')
pr_shows

In [ ]:
# Como serão considerados apenas shows grandes e médios para a análise do trabalho, podemos considerar praticamente impossível o mesmo artista fazer um show em um mesmo local em um mesmo dia. Assim, serão apagadas as linhas duplicadas dessa base.

shows = shows.drop_duplicates()

# Também apagarei shows que ocorrerão em outros municípios que não São Paulo capital.

shows = shows[shows['cidade'] == 'São Paulo']

# Essa é uma base de dados fato, portanto os conjuntos de valores nela são praticamente únicos e apenas retirarei da análise as colunas de estado e cidade, que utilizei apenas para conferência na construção. Como normalizei o nome do local, também o apagarei não normalizado.

shows = shows.drop(['estado', 'cidade', 'local'], axis = 1)

shows.to_csv('../data/raw/shows_limpo.csv', index=True)

In [ ]:
shows.info()

In [ ]:
shows.head()

## Base de Artistas
Criada com o API do Spotify, rodei o código 4 vezes para incremento.

In [ ]:
with open('../data/external/artistas.csv', 'rb') as f:
    enc = chardet.detect(f.read())
    print(enc)
    
artistas = pd.read_csv('../data/external/artistas.csv', sep = ';')
print(artistas.info())
artistas.head()

In [ ]:
artistas['nome'] = artistas['nome'].apply(normalize)
artistas['generos'] = artistas['generos'].apply(normalize)

#Os dados de gêneros em branco foram corrigidos com ajuda de ia manualmente.

pr_artistas = ProfileReport(artistas, title='Profiling Report of Artistas Data Base')
pr_artistas

#Assim como a base anterior, essa base é única.

## Base de Calendário

In [ ]:
calend = pd.read_csv('../data/external/calendario_sp.csv')

print(calend.info())
calend.head()

In [ ]:
# Os dados em branco da coluna 'nome_feriado' não precisam ser tratados, uma vez que dependem de existir um feriado na data e, caso não o haja, ela realmente ficará em branco. Entretanto, os dados referentes ao clima podemos substituir separando em dois grupos de dados: o primeiro referente a dados numéricos de temperatura, precipitação e o código, podemos gerenciar pela média em relação a anos anteriores, considerando períodos mensais semelhantes; já em relação à descrição, podemos fazê-la considerando o resultado da média do código calculada, utilizando-se a moda para cálculo.

calend['data'] = pd.to_datetime(calend['data'], format='%Y-%m-%d', errors='coerce')
calend['num_mes'] = calend['data'].dt.month

meses = {
    1: 'JANEIRO',
    2: 'FEVEREIRO',
    3: 'MARÇO',
    4: 'ABRIL',
    5: 'MAIO',
    6: 'JUNHO',
    7: 'JULHO',
    8: 'AGOSTO',
    9: 'SETEMBRO',
    10: 'OUTUBRO',
    11: 'NOVEMBRO',
    12: 'DEZEMBRO'
}
calend['mes'] = calend['num_mes'].map(meses)

calend.info()

In [ ]:
cols = ['temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'weathercode']

calend[cols] = calend.groupby(['mes'])[cols].transform(lambda x: x.fillna(x.mean()))

calend['temperatura_media'] = (calend['temperature_2m_max'] + calend['temperature_2m_min'])/2
calend = calend.drop(['temperature_2m_max','temperature_2m_min'], axis = 1)

calend['descricao_clima'] = calend.groupby(['weathercode'])['descricao_clima'].fillna(pd.Series.mode)
calend['dia_semana'] = calend['dia_semana'].apply(normalize)
calend['nome_feriado'] = calend['nome_feriado'].apply(normalize)
calend['tipo_dia'] = calend['tipo_dia'].apply(normalize)
calend['descricao_clima'] = calend['descricao_clima'].apply(normalize)

calend.info()

In [ ]:
pr_calend = ProfileReport(calend, title='Profiling Report of Calendario Data Base')
pr_calend

In [ ]:
#Dimensaionalisar a base de dados (data tem mais de uma entrada)
calend = (
    calend
    .groupby(['data', 'ano'], as_index=False)
    .agg({
        'dia_semana': 'first',
        'nome_feriado': 'first',
        'tipo_dia': 'first',
        'precipitation_sum': 'mean',
        'weathercode': 'first',
        'descricao_clima': 'first',
        'num_mes': 'first',
        'mes': 'first', 
        'temperatura_media': 'mean'
    })
)

In [ ]:
# Como são muitas variáveis categóricas utilizarei o método de análise phik

phik_corr = calend.phik_matrix(interval_cols=[])

plt.figure(figsize=(14, 12))
sns.heatmap(
    phik_corr,
    annot=True,
    fmt='.2f',
    cmap='crest',
    center=0,
    linewidths=2,
    linecolor='#FFFFFF'
)
plt.title('Matriz de Correlação (Phik)', fontsize=14)
plt.show()

In [ ]:
# Excluirei o nome do feriado, uma vez que entendo que as variáveis tipo_dia e num_mes estão duplicando. Entre as variáveis de clima, percebemos uma forte relação entre todas, considerando wethercode e descricao_clima as de melhor entendimento. Manterei apenas a segunda para análise, uma vez que essa dá significado em palavras para os números da primeira.

calend = calend.drop(['nome_feriado', 'precipitation_sum', 'weathercode', 'num_mes', 'temperatura_media'], axis = 1)

calend.to_csv('../data/raw/calendario_tratado.csv', index=True)

## Base de Setores

In [ ]:
setores = pd.read_csv('../data/external/setores.csv', sep = ';', encoding= 'iso8859')

print(setores.info())
setores.head()

In [ ]:
setores['Categoria_Setor'].unique()

setores['Nome_Setor'] = setores['Nome_Setor'].apply(normalize)
setores['Categoria_Setor'] = setores['Categoria_Setor'].apply(normalize)

# Base de dados criada manualmente, e, portanto, não precisa de mais tratamento.

## Base de Festivias

Base criada manualmente para enriquecer a análise.

In [ ]:
fest = pd.read_csv('../data/external/festivais.csv', sep = ';', encoding = 'iso8859')

print(fest.info())
fest.head()

In [ ]:
fest['nome_local'] = fest['Local'].apply(normalize)
fest['festival'] = fest['Festival'].apply(normalize)
fest['headliners'] = fest['Headliners'].apply(normalize)

fest = fest.dropna(subset=['Ano'])
fest['ano'] = fest['Ano'].astype(int)

# Transformar a data para o formato correto
fest['data'] = pd.to_datetime(fest['Data '], dayfirst=True, errors='coerce')

fest.head()

In [ ]:
fest = fest.drop(['Local', 'Festival', 'Headliners', 'Data ', 'Ano'], axis = 1)

# Nessa base eu apenas apaguei a coluna original de local, como tratamento.
# Utilizarei essa base inicialmente para retirar os shows que aconteceram durante o jogo da NFL no Brasil, uma vez que o público nestes eventos não tem como principal foco o show musical, mas sim o esportivo.

# -------------------------------------------------------------------------------------
# Tratar e transformar as bases carregadas

## Tratamento inicial para a base de Shows
Criação da base de dados de shows com a interação entre essa base e a base de locais para seleção dos shows em locais com público-alvo maior ou igual a 2000 presentes (shows médios e grandes).

In [ ]:
shows_locais = shows.merge(locais, how ='inner', on= 'nome_local')
shows_locais.head()

In [ ]:
shows_locais.info()
shows_locais.to_csv('../data/raw/shows_locais.csv', index=True)

In [ ]:
# Para reduzir entradas outliers na base anterior, como shows de artistas com poucos seguidores ou baixa popularidade em locais desproporcionalmente grandes, significando, provavelmente, uma abertura para um show de uma banda maior ou um festival. Farei um comparativo entre a base de artistas e a anterior, focando em shows no mesmo dia e no mesmo local, mantendo os artistas com mais alta de popularidade.

shows_locais_artistas = shows_locais.merge(artistas, how ='left', on = 'nome')

shows_tratado = (shows_locais_artistas
                 .sort_values(by = ['seguidores', 'popularidade'], ascending=False)
                 .drop_duplicates(subset=['nome', 'data', 'nome_local'], keep='first')
                 .copy())

shows_tratado.info()

In [ ]:
shows_tratado.head()

In [ ]:
shows_tratado = shows_tratado.merge(fest, how = 'left', on = ['data', 'nome_local', 'ano'])

shows_tratado.info()

In [ ]:
# Apagando eventos não musicais
shows_tratado = shows_tratado[~shows_tratado['festival'].isin(['NFL BRASIL', 'GP FORMULA 1', 'FESTIVAL DE MOTOS DE INTERLAGOS', 'SAO PAULO HORROR EXPO', 'ROLEX 6 HORAS', 'FINAL DA KINGS LEAGUE BRASIL', 'S.I.N. FESTIVAL', 'FINAL DO CAMPEONATO PAULISTA', 'TIKTOK AWARDS'])]

# Para o tratamento dos dados faltosos vindos do API do Spotify, pegarei a listagem de nomes e puxarei esses dados usando o mesmo código.
sem_dados = shows_tratado[shows_tratado['generos'].isnull()]['artista'].unique()
consulta = []
for i in sem_dados:
    cons = f'{i}'
    consulta.append(cons)

print(consulta)

In [ ]:
CLIENTES = [ ('**', '**') ]
LIMIT = 50
MAX_PAGES = 10
PARCIAL = 'parcial_trat.json'
CSV_FINAL = 'artistas_trat.csv'

def autenticar(indice):
    cid, secret = CLIENTES[indice]
    print(f'\n🔐 Iniciando API ID: {cid[:10]}...')
    return Spotify( auth_manager=SpotifyClientCredentials( client_id=cid, client_secret=secret ) )


def load_parcial():
    try:
        with open(PARCIAL, 'r', encoding='utf-8') as f:
            todos = json.load(f)
        print(f'🟢 Parcial carregada: {len(todos)} registros')
    except:
        todos = []
        print('⚪ Nenhuma parcial encontrada. Iniciando nova coleta.')
    return todos

def save_parcial(todos):
    with open(PARCIAL, 'w', encoding='utf-8') as f:
        json.dump(todos, f, ensure_ascii=False, indent=2)

def extract_from_item(a, fonte):
    followers = a.get('followers', {}).get('total', 0)
    genres = ', '.join(a.get('genres', []))
    images = a.get('images', [])
    image_url = images[0]['url'] if images else ''
    return { 'nome': a.get('name', ''),
             'id_spotify': a.get('id', ''),
             'generos': genres,
             'popularidade': a.get('popularity', 0),
             'seguidores': followers,
             'imagem': image_url,
             'fonte': fonte }

def coletar(sp, query_list, fonte_label):
    todos = load_parcial()
    for termo in query_list:
        for pag in range(MAX_PAGES):
            try:
                r = sp.search( q=termo,
                               type='artist',
                               limit=LIMIT,
                               offset=pag * LIMIT )
                items = r.get('artists', {}).get('items', [])

                if not items:
                    break

                for a in items:
                    seg = a.get('followers', {}).get('total', 0)
                    todos.append(extract_from_item(a, fonte_label))
                    if len(todos) % 50 == 0:
                        save_parcial(todos)

                time.sleep(random.uniform(1.3, 2.4))

            except Exception as e:
                time.sleep(8)

    save_parcial(todos)
    print('✔ Coleta concluída.')

    return todos


if __name__ == '__main__':
    sp = autenticar(0)
    coletar(sp, [f'nome:'{g}'' for g in consulta], 'nova consulta')

    print('\n📦 Gerando CSV...')
    todos = load_parcial()
    df = pd.DataFrame(todos).drop_duplicates(subset='id_spotify')

    df.to_csv(CSV_FINAL, index=False, encoding='utf-8-sig')
    print(f'🎉 CSV FINAL SALVO! Total de artistas únicos: {len(df)}')
    print('Arquivo:', CSV_FINAL)

In [ ]:
df = pd.read_csv('../data/external/artistas_trat.csv', sep = ',')
df.tail(2)

In [ ]:
df['nome'] = df['nome'].apply(normalize)
df['generos'] = df['generos'].apply(normalize)

df.info()

In [ ]:
df = df.dropna(subset=['nome'])

df = df.set_index('nome')

df = df.reset_index()

df = df.drop_duplicates(subset='id_spotify').sort_values('nome')

In [ ]:
shows_tratado = shows_tratado.merge(df, on='nome', how='left', suffixes=('', '_spotify'))

In [ ]:
cols = ['generos', 'popularidade', 'seguidores', 'imagem', 'id_spotify', 'fonte']

for c in cols:
    shows_tratado[c] = shows_tratado[c].fillna(shows_tratado[f'{c}_spotify'])

shows_tratado = shows_tratado.drop(columns=[f'{c}_spotify' for c in cols])

In [ ]:
shows_tratado = shows_tratado[(shows_tratado['popularidade'] > 59) & (shows_tratado['seguidores'] >= 150000)]

shows_tratado.info()

In [ ]:
shows_tratado = shows_tratado.dropna(subset=['fonte'])
shows_tratado['festival'] = shows_tratado['festival'].fillna(False)
shows_tratado['headliners'] = shows_tratado['headliners'].fillna(False)

shows_tratado.head()

In [ ]:
shows_tratado.to_csv('../data/raw/shows_tratado.csv', index=True)

# Com essa nova base filtrada, podemos criar a base de dados com os preços de ingressos por setor e pegar a lotação,quando disponível dos shows de interesse.

# --------------------------------------------------------------------------------------
# Bases de dados desenvolvidas a partir do resultado do primeiro tratamento

## Lotação com shows

Essa base foi criada a partir dos acima obtidos, manualmente, considerando informações divulgadas por reportagens ou pelas produtoras.

In [ ]:
lot = pd.read_csv('../data/external/shows_lotacao.csv', sep=';')

print(lot.info())
lot.head()

In [ ]:
lot['data'] = pd.to_datetime(lot['data'], dayfirst=True, errors='coerce')
lot['lotacao'] = lot['lotacao'].astype('float64')

lot['nome'] = lot['nome'].apply(normalize)
lot['nome_local'] = lot['nome_local'].apply(normalize)

## Valores por setor por show
Essa base foi criada a partir dos acima obtidos, manualmente, considerando informações divulgadas por reportagens ou por sites de vendas de ingressos.

In [ ]:
val = pd.read_csv('../data/external/valores.csv', sep = ';', encoding = 'iso8859')

print(val.info())
val.head()

In [ ]:
col_set = val.columns[4:]
val = val.replace(',', '.', regex=True)

val[col_set] = val[col_set].astype('float64')

# Para inclusão dos valores NA (não 0) utilizarei a moda entre setores por local, e ano
moda = (
    val.groupby(['nome_local', 'data'])[col_set]
      .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
      .reset_index()
)

val[col_set] = val[col_set].fillna(moda[col_set])

In [ ]:
val.info()

In [ ]:
val.head()

In [ ]:
# rotacionar as colunas de setores e por show
val = pd.melt(
    val,
    id_vars=['nome', 'data', 'ano', 'nome_local'],
    var_name='setor',
    value_name='preco'
)

# Shows gratuitos geram anomalias em relação ao número estrtimado de participantes
val = val[val['preco'] != 0]
val = val.dropna()

val['preco_medio'] = (val.groupby(['nome', 'nome_local', 'data'])['preco'].transform('mean')) 

val.info()

In [ ]:
val.head()

In [ ]:
val['nome'] = val['nome'].apply(normalize)
val['nome_local'] = val['nome_local'].apply(normalize)
val['nome_setores'] = val['setor'].apply(normalize)
val['data'] = pd.to_datetime(val['data'], dayfirst=True, errors='coerce')

val['preco'] = val['preco'].apply(lambda c: pd.to_numeric(c, errors='coerce'))
val = val[val['preco'] != 0]

val.info()

In [ ]:
val.head()

In [ ]:
val.to_csv('../data/raw/valores.csv', index=True)

# -------------------------------------------------------------------------------------------------
# Desenvolvimento das bases de dados unificadas utilizadas no projeto

## Base Análise

In [ ]:
base_final = shows_tratado.merge(lot, how = 'left', on = ['nome', 'data', 'ano', 'nome_local'])
base_final.info()

In [ ]:
base_final['generos'] = base_final['generos'].apply(
    lambda x: x.split(',')[0] if isinstance(x, str) else x
)
base_final['generos'] = base_final['generos'].apply(normalize)

base_final['lotacao'] = base_final.groupby(['nome_local', 'festival', 'generos'])['lotacao'].transform(lambda x: x.fillna(x.mean()))

In [ ]:
base_final = base_final.merge(val, how = 'left', on = ['nome', 'data', 'ano', 'nome_local'])
base_final.info()

In [ ]:
base_final = base_final.merge(setores, how = 'left', left_on = ['nome_setores'], right_on = ['Nome_Setor'])
base_final.info()

In [ ]:
base_final = base_final.merge(calend, how = 'left', on = ['data', 'ano'])

base_final['festival'] = base_final['festival'].fillna(False)
    
base_final.info()

In [ ]:
# 2021 as casas estavam operando com restrições portanto considerar esses shows pode gerar anomalias
base_final = base_final[base_final['ano'] > 2021]
base_final['lotacao_pct'] = base_final['lotacao']/base_final['Capacidade']

In [ ]:
base_final.head()

In [ ]:
# Desconsiderar shows com uma lotação muito maior que a capacidade, sabendo que a capacidade da casa depende da configiiuração do show o valor fornecido oficialmente pode variar 
#para mais ou menos mas o número de presentes não pode ser tão absurdamente maior, adortarei um valor de 150% (1,5) a capaciidade
base_final = base_final[base_final['lotacao_pct'] < 1.5]

base_final.info()

In [ ]:
base_final = base_final.dropna(subset=['preco'])
base_final = base_final.fillna(False)

base_final.info()

In [ ]:
base_final.to_csv('../data/temp/base_tamanhooriginal.csv', index=True, encoding='utf-8-sig')

In [ ]:
#Limpando as colunas que não serão essenciais para a anãlise e modelagem dos dados
base_final = base_final.drop(
    ['artista', # manterei apenas os nomes normalizados nessa base
     'imagem',   # Essa coluna será utilizada com outro objetivo não para a análise dos dados apresentados
     'id_spotify',  # Coluna de controle e insertção em banco de dados estruturado
     'Setor_id',   # Coluna para controle e inserção em banco de dados estruturado
     'fonte',   # Coluna de controle
     'headliners',
     'nome_setores',
     'Nome_Setor',
     'Capacidade'   # Como mencionado anteriormente
     ], axis = 1)

In [ ]:
base_final['arrecadacao_estimada'] = base_final['preco'] * base_final['lotacao']

# Calcule a distância em dias para o show anterior do mesmo gênero
base_final['distancia_dias_anterior'] = base_final.groupby('generos')['data'].diff().dt.days
base_final['distancia_dias_anterior'] = base_final['distancia_dias_anterior'].fillna(999)

base_final['ano'] = base_final['ano'].astype('string')
base_final = base_final.drop_duplicates()

base_final.info()

base_final.to_csv('../data/processed/base_projeto.csv', index=False, encoding='utf-8-sig')

## Base unificada de artistas

In [ ]:
df.info()

In [ ]:
artistas.info()

In [ ]:
artistas_final = pd.concat([artistas, df])

artistas_final['generos'] =  artistas_final['generos'].apply(
    lambda x: x.split(',')[0] if isinstance(x, str) else x
)
artistas_final['generos'] = artistas_final['generos'].apply(normalize)

artistas_final = artistas_final.drop_duplicates()

artistas_final.info()

artistas_final = artistas_final.dropna(subset=['generos'])
artistas_final['imagem'] = artistas_final['imagem'].fillna('')

In [ ]:
artistas_final = artistas_final.drop(['fonte', 'id_spotify'], axis = 1)
artistas_final = artistas_final[(artistas_final['popularidade'] > 59) & (artistas_final['seguidores'] >= 150000)]

artistas_final.info()

In [ ]:
artistas_final.to_csv('../data/temp/artistas_projeto.csv', index=False, encoding='utf-8-sig')